# 08. Molecular Dynamics Trajectories & Radial Distribution Functions

`Trajectory.rmsd()` unwraps periodic boundary crossings cumulatively across consecutive steps before optional Kabsch alignment, and `Trajectory.compute_rdf()` integrates **exact spherical-shell volumes** across periodic image shells up to any requested `r_max` (with optional `strict=True` Wigner-Seitz minimum-image enforcement).

## Basic Usage

Plot RMSD evolution from a bundled LAMMPS `.lammpstrj` trajectory (`cu_md.lammpstrj`) alongside a GROMACS liquid-water O–O RDF (`water_rdf.xvg`):

In [ ]:
import matplotlib.pyplot as plt
import xkcd_matchem as xm

md = xm.load_trajectory("cu_md.lammpstrj")
rdf_xvg = xm.load("water_rdf.xvg")

with xm.xkcd_style("classic"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.2, 4.4))
    xm.plot_md_timeseries(md, quantity="rmsd", running_window=3, ax=ax1, title="Cu MD RMSD Trace")
    xm.plot_rdf(rdf_xvg.x, rdf_xvg.y, unit="nm", ax=ax2, title="Liquid Water O-O RDF (XVG)")
    fig.tight_layout()

## Advanced Usage

Compute $g_{\text{Cu-Cu}}(r)$ directly from periodic atomic frames while respecting the Wigner-Seitz minimum-image inradius (`max_min_image_radius`), and explore `rmsd(indices=...)` on atom subsets:

In [ ]:
r_ws = xm.core.geometry.max_min_image_radius(md[0].cell, md[0].pbc)
r_safe = round(0.95 * r_ws, 2)
print(f"Wigner-Seitz inradius: {r_ws:.2f} A -> computing strict g(r) up to {r_safe} A")

r_vals, g_vals = md.compute_rdf(("Cu", "Cu"), r_max=r_safe, n_bins=30, strict=True)
sub_rmsd = md.rmsd(ref_index=0, indices=[0, 1, 2, 3])
print("4-atom subset RMSD max (A):", round(float(sub_rmsd.max()), 4))

ax = xm.plot_rdf(r_vals, g_vals, unit="A", color="#9C755F", title="Computed Cu-Cu Periodic RDF (strict=True)")